## EDA + featue selection

In [6]:
# EDA ON CHUNKS — SUMMARY STATISTICS (NO CONCATENATION)
# ============================================================================
# Computes per-column statistics from merged_data_final.csv in a single
# chunked pass, with no full-file load into memory. Uses reservoir sampling
# for unbiased percentile estimates across the full 400M-row dataset.
# ============================================================================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import random

CSV_PATH = "merged_output_hourly_imerg/merged_data_final.csv"
OUTPUT_DIR = "eda_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

EXCLUDE_COLS = {"time", "date", "latitude", "longitude",
                "era5_lat_idx", "era5_lon_idx"}
HIST_BINS = 60
SAMPLE_SIZE = 200_000
CHUNK_SIZE = 500_000

print(f"Source: {CSV_PATH} ({os.path.getsize(CSV_PATH)/1e9:.2f} GB)")
print("Computing statistics from CSV (chunked, no concatenation)...\n")

stats_dict = {}
total_rows = 0
rng = random.Random(42)

for i, chunk in enumerate(pd.read_csv(CSV_PATH, chunksize=CHUNK_SIZE,
                                        on_bad_lines="skip")):
    total_rows += len(chunk)
    if (i + 1) % 20 == 0:
        print(f"  ... {total_rows:,} rows processed")

    for col in chunk.columns:
        if col in EXCLUDE_COLS or not pd.api.types.is_numeric_dtype(chunk[col]):
            continue
        if col not in stats_dict:
            stats_dict[col] = {
                "count": 0, "sum": 0.0, "sum_sq": 0.0,
                "min": np.inf, "max": -np.inf, "missing": 0,
                "reservoir": [], "seen": 0,
            }
        s = chunk[col]
        valid = s.dropna()
        d = stats_dict[col]
        d["missing"] += int(s.isna().sum())

        if len(valid) == 0:
            continue

        d["count"] += len(valid)
        d["sum"] += float(valid.sum())
        d["sum_sq"] += float((valid ** 2).sum())
        d["min"] = min(d["min"], float(valid.min()))
        d["max"] = max(d["max"], float(valid.max()))

        vals = valid.to_numpy()
        for v in vals:
            d["seen"] += 1
            if len(d["reservoir"]) < SAMPLE_SIZE:
                d["reservoir"].append(float(v))
            else:
                j = rng.randint(0, d["seen"] - 1)
                if j < SAMPLE_SIZE:
                    d["reservoir"][j] = float(v)

print(f"\n✓ Scanned {total_rows:,} rows. Building summary table...")

rows = []
for col, d in stats_dict.items():
    count = d["count"]
    mean = d["sum"] / count if count > 0 else np.nan
    var = (d["sum_sq"] / count - mean ** 2) if count > 0 else np.nan
    std = float(np.sqrt(max(var, 0))) if count > 0 else np.nan

    res = np.sort(np.array(d["reservoir"])) if d["reservoir"] else np.array([])
    p25 = float(np.percentile(res, 25)) if len(res) > 0 else np.nan
    p50 = float(np.percentile(res, 50)) if len(res) > 0 else np.nan
    p75 = float(np.percentile(res, 75)) if len(res) > 0 else np.nan
    p95 = float(np.percentile(res, 95)) if len(res) > 0 else np.nan

    rows.append({
        "attribute":  col,
        "count":      count,
        "missing":    int(d["missing"]),
        "missing_%":  round(100 * d["missing"] / max(count + d["missing"], 1), 3),
        "min":        round(d["min"], 4) if count > 0 else np.nan,
        "p25":        round(p25, 4),
        "median":     round(p50, 4),
        "mean":       round(mean, 4) if count > 0 else np.nan,
        "p75":        round(p75, 4),
        "p95":        round(p95, 4),
        "max":        round(d["max"], 4) if count > 0 else np.nan,
        "std":        round(std, 4) if count > 0 else np.nan,
    })

stats_df = pd.DataFrame(rows).sort_values("attribute").reset_index(drop=True)

stats_csv = os.path.join(OUTPUT_DIR, "summary_statistics.csv")
stats_df.to_csv(stats_csv, index=False, float_format="%.4f")
print(f"✓ Saved summary statistics -> {stats_csv}")
print(f"\n{stats_df.to_string(index=False)}")

print("\nGenerating summary table image...")
fig, ax = plt.subplots(figsize=(min(2 + 1.1 * stats_df.shape[1], 22),
                                 0.5 + 0.4 * stats_df.shape[0]))
ax.axis("off")
tbl = ax.table(
    cellText=stats_df.values,
    colLabels=stats_df.columns,
    cellLoc="center",
    loc="center",
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(7)
tbl.scale(1, 1.3)
for (r, c), cell in tbl.get_celld().items():
    if r == 0:
        cell.set_facecolor("#34568B")
        cell.set_text_props(color="white", fontweight="bold")
    elif r % 2 == 0:
        cell.set_facecolor("#f0f3f7")

ax.set_title("Summary statistics per attribute", fontsize=12, pad=12)
fig.tight_layout()
table_path = os.path.join(OUTPUT_DIR, "summary_statistics.png")
fig.savefig(table_path, dpi=140, bbox_inches="tight")
plt.close(fig)
print(f"✓ Saved summary table image -> {table_path}")
print(f"\n✓ EDA complete. Outputs in: {OUTPUT_DIR}/")

Source: merged_output_hourly_imerg/merged_data_final.csv (89.18 GB)
Computing statistics from CSV (chunked, no concatenation)...

  ... 10,000,000 rows processed
  ... 20,000,000 rows processed
  ... 30,000,000 rows processed
  ... 40,000,000 rows processed
  ... 50,000,000 rows processed
  ... 60,000,000 rows processed
  ... 70,000,000 rows processed
  ... 80,000,000 rows processed
  ... 90,000,000 rows processed
  ... 100,000,000 rows processed
  ... 110,000,000 rows processed
  ... 120,000,000 rows processed
  ... 130,000,000 rows processed
  ... 140,000,000 rows processed
  ... 150,000,000 rows processed
  ... 160,000,000 rows processed
  ... 170,000,000 rows processed
  ... 180,000,000 rows processed
  ... 190,000,000 rows processed
  ... 200,000,000 rows processed
  ... 210,000,000 rows processed
  ... 220,000,000 rows processed
  ... 230,000,000 rows processed
  ... 240,000,000 rows processed
  ... 250,000,000 rows processed
  ... 260,000,000 rows processed
  ... 270,000,000 row

In [8]:
import pandas as pd

path = "merged_output_hourly_imerg/merged_data_final.csv"
CHUNK_SIZE = 1_000_000

# Check what ERA5 columns are actually present first
header = pd.read_csv(path, nrows=0).columns.tolist()
era5_cols = [c for c in header if c.startswith("era5_")]
print(f"ERA5 columns present: {era5_cols}")

# Pick the first one to check NaN pattern by latitude
test_col = era5_cols[0]
print(f"\nChecking NaN pattern using '{test_col}'...")

nan_by_lat = {}
for chunk in pd.read_csv(path, usecols=["latitude", test_col],
                          chunksize=CHUNK_SIZE, on_bad_lines="skip"):
    nan_rows = chunk[chunk[test_col].isna()]
    for lat, grp in nan_rows.groupby("latitude"):
        nan_by_lat[lat] = nan_by_lat.get(lat, 0) + len(grp)

print("\nLatitudes with ERA5 NaN values:")
for lat in sorted(nan_by_lat.keys()):
    print(f"  {lat:.4f}°N: {nan_by_lat[lat]:,} NaN rows")
print(f"\nTotal distinct latitudes with NaN: {len(nan_by_lat)}")

ERA5 columns present: ['era5_d2m', 'era5_t2m', 'era5_msl', 'era5_sst', 'era5_sp', 'era5_skt', 'era5_swvl1', 'era5_z', 'era5_cape', 'era5_tcwv', 'era5_blh', 'era5_cp', 'era5_swvl2', 'era5_swvl3', 'era5_slt', 'era5_lsm']

Checking NaN pattern using 'era5_d2m'...

Latitudes with ERA5 NaN values:
  62.5000°N: 2,867,328 NaN rows

Total distinct latitudes with NaN: 1


In [9]:
# DROP LATITUDE 62.5°N BORDER ROW — IN-PLACE, MEMORY-SAFE, CHUNKED
# ============================================================================
# The northernmost latitude row (62.5°N) has 0.709% NaN across all ERA5
# variables due to a boundary interpolation artifact: linear interpolation
# cannot extrapolate past the outermost source data point, leaving this row
# permanently empty. Confirmed as the ONLY affected latitude (single distinct
# latitude in the NaN-by-lat check). Dropping it reduces the grid from
# 141x131 to 140x131 = 18,340 cells per hour, and total rows from
# 404,293,248 to 401,425,920.
# ============================================================================
import os
import pandas as pd

TARGET_PATH = "merged_output_hourly_imerg/merged_data_final.csv"
TEMP_PATH = "merged_output_hourly_imerg/merged_data_final.tmp.csv"
CHUNK_SIZE = 1_000_000

DROP_LAT = 62.5
LAT_TOL = 0.001

print(f"Source: {TARGET_PATH} ({os.path.getsize(TARGET_PATH)/1e9:.2f} GB)")
print(f"Dropping all rows where latitude is {DROP_LAT}°N (±{LAT_TOL}°)\n")

n_rows_in = 0
n_rows_out = 0
n_dropped = 0
first_chunk = True

reader = pd.read_csv(TARGET_PATH, chunksize=CHUNK_SIZE, on_bad_lines="skip")
for i, chunk in enumerate(reader):
    n_rows_in += len(chunk)
    keep = ~(chunk["latitude"] - DROP_LAT).abs().lt(LAT_TOL)
    n_dropped += int((~keep).sum())
    chunk = chunk.loc[keep]
    n_rows_out += len(chunk)

    chunk.to_csv(TEMP_PATH, mode="w" if first_chunk else "a",
                 header=first_chunk, index=False)
    first_chunk = False

    if (i + 1) % 20 == 0:
        print(f"  ... {n_rows_in:,} rows read, {n_dropped:,} dropped so far")

print(f"\n✓ Complete")
print(f"  Rows in:      {n_rows_in:,}")
print(f"  Rows dropped: {n_dropped:,}")
print(f"  Rows out:     {n_rows_out:,}")

EXPECTED_DROPPED = 131 * 21888
if n_dropped != EXPECTED_DROPPED:
    print(f"\n⚠ Expected to drop {EXPECTED_DROPPED:,} rows but dropped {n_dropped:,}.")
    print("  NOT replacing source — investigate before proceeding.")
    os.remove(TEMP_PATH)
else:
    print(f"✓ Drop count matches expected ({EXPECTED_DROPPED:,})")
    os.replace(TEMP_PATH, TARGET_PATH)
    print(f"✓ Replaced {TARGET_PATH} in place")
    print(f"  New file size: {os.path.getsize(TARGET_PATH)/1e9:.2f} GB")
    print(f"  New grid: 140 lat x 131 lon = 18,340 cells per hour")
    print(f"  New total rows: {n_rows_out:,}")

Source: merged_output_hourly_imerg/merged_data_final.csv (89.18 GB)
Dropping all rows where latitude is 62.5°N (±0.001°)

  ... 20,000,000 rows read, 141,742 dropped so far
  ... 40,000,000 rows read, 283,615 dropped so far
  ... 60,000,000 rows read, 425,488 dropped so far
  ... 80,000,000 rows read, 567,361 dropped so far
  ... 100,000,000 rows read, 709,103 dropped so far
  ... 120,000,000 rows read, 850,976 dropped so far
  ... 140,000,000 rows read, 992,849 dropped so far
  ... 160,000,000 rows read, 1,134,722 dropped so far
  ... 180,000,000 rows read, 1,276,595 dropped so far
  ... 200,000,000 rows read, 1,418,337 dropped so far
  ... 220,000,000 rows read, 1,560,210 dropped so far
  ... 240,000,000 rows read, 1,702,083 dropped so far
  ... 260,000,000 rows read, 1,843,956 dropped so far
  ... 280,000,000 rows read, 1,985,698 dropped so far
  ... 300,000,000 rows read, 2,127,571 dropped so far
  ... 320,000,000 rows read, 2,269,444 dropped so far
  ... 340,000,000 rows read, 2,4

In [1]:
# TIME CONTINUITY CHECK PER GRID CELL
# Confirms whether each 0.1° grid cell has a gap-free hourly time series.
# This matters before building 6h-ahead labels: a naive shift(6) per cell is
# only valid if there really are no missing hours in between — otherwise
# "6 rows later" silently stops meaning "6 hours later" for some cells.
import os
import time
import numpy as np
import pandas as pd

CLEANED_DIR = "merged_output_hourly_imerg"
CSV_PATH = os.path.join(CLEANED_DIR, "merged_data_with_windspeed.csv")

UK_BOUNDS = {"lat_min": 49.9, "lat_max": 60.9, "lon_min": -8.2, "lon_max": 1.8}
GRID_RES = 0.1
CHUNK_SIZE = 1_000_000

n_lat = int(round((UK_BOUNDS["lat_max"] - UK_BOUNDS["lat_min"]) / GRID_RES)) + 1
n_lon = int(round((UK_BOUNDS["lon_max"] - UK_BOUNDS["lon_min"]) / GRID_RES)) + 1
# NOTE: +1 is required — an inclusive range from min to max with fixed step
# has (max-min)/step + 1 distinct points, not (max-min)/step. Missing this
# previously caused the true last row/column to be clipped into the
# second-to-last, silently merging two distinct grid columns into one.

# Accumulate sorted-unique hour-index sets per grid cell, keyed by integer
# (lat_idx, lon_idx) — same rounding approach as the earlier binning fix,
# so floating-point noise can't split one true grid cell into two keys.
cell_hours = {}

print(f"Scanning {CSV_PATH} for time continuity per grid cell...")
t0 = time.time()
n_rows = 0

reader = pd.read_csv(
    CSV_PATH,
    usecols=["time", "latitude", "longitude"],
    chunksize=CHUNK_SIZE,
    on_bad_lines="skip",
)

for i, chunk in enumerate(reader):
    chunk = chunk.dropna(subset=["time", "latitude", "longitude"])
    if len(chunk) == 0:
        continue
    n_rows += len(chunk)

    lat = chunk["latitude"].to_numpy(dtype=np.float64)
    lon = chunk["longitude"].to_numpy(dtype=np.float64)
    lat_idx = np.clip(np.round((lat - UK_BOUNDS["lat_min"]) / GRID_RES).astype(np.int64), 0, n_lat - 1)
    lon_idx = np.clip(np.round((lon - UK_BOUNDS["lon_min"]) / GRID_RES).astype(np.int64), 0, n_lon - 1)
    cell_key = lat_idx * n_lon + lon_idx  # single integer key per grid cell

    # Some rows have date-only strings (likely midnight timestamps that lost
    # their "00:00:00" suffix somewhere in the export), while most rows have
    # a full "date time" string — pandas can't infer one fixed format across
    # both. Normalize date-only strings before parsing, which is much faster
    # than format="mixed" on 32GB of data.
    time_str = chunk["time"].astype(str)
    date_only = time_str.str.len() == 10  # "YYYY-MM-DD" is exactly 10 chars
    time_str = time_str.where(~date_only, time_str + " 00:00:00")
    # FIXED: previously assumed .astype('int64') always gives nanoseconds,
    # but pandas 2.x may parse into second/microsecond resolution instead —
    # this silently produced hour values ~1000x too small, collapsing many
    # real hours together and invalidating the original "gap-free" result.
    parsed_t = pd.to_datetime(time_str, format="%Y-%m-%d %H:%M:%S")
    t_hours = ((parsed_t - pd.Timestamp("1970-01-01")) // pd.Timedelta(hours=1)).to_numpy(dtype=np.int64)

    df = pd.DataFrame({"cell": cell_key, "hour": t_hours})
    for key, group in df.groupby("cell", sort=False)["hour"]:
        cell_hours.setdefault(key, []).append(group.to_numpy())

    if (i + 1) % 20 == 0:
        elapsed = time.time() - t0
        print(f"  Scanned {i+1} chunks, {n_rows:,} rows, {elapsed:,.1f}s elapsed")

print(f"✓ Scanned {n_rows:,} rows in {time.time() - t0:,.1f}s. Checking continuity...")

n_cells = 0
n_fully_continuous = 0
gap_size_counts = {}
cells_with_gaps = 0

for key, arrays in cell_hours.items():
    hours = np.unique(np.concatenate(arrays))  # sorted unique hours for this cell
    n_cells += 1
    if len(hours) < 2:
        continue
    diffs = np.diff(hours)
    if np.all(diffs == 1):
        n_fully_continuous += 1
    else:
        cells_with_gaps += 1
        for gap in diffs[diffs != 1]:
            gap_size_counts[int(gap)] = gap_size_counts.get(int(gap), 0) + 1

print(f"\nGrid cells with any data: {n_cells:,}")
print(f"Fully continuous (1h steps, no gaps): {n_fully_continuous:,} ({100*n_fully_continuous/max(n_cells,1):.1f}%)")
print(f"Cells with at least one gap: {cells_with_gaps:,} ({100*cells_with_gaps/max(n_cells,1):.1f}%)")
print("\nGap size distribution (hours skipped, top 15):")
for gap, cnt in sorted(gap_size_counts.items(), key=lambda x: -x[1])[:15]:
    print(f"  {gap}h gap: {cnt:,} occurrences")

Scanning merged_output_hourly_imerg/merged_data_with_windspeed.csv for time continuity per grid cell...
  Scanned 20 chunks, 20,000,000 rows, 13.4s elapsed
  Scanned 40 chunks, 40,000,000 rows, 26.3s elapsed
  Scanned 60 chunks, 60,000,000 rows, 39.6s elapsed
  Scanned 80 chunks, 80,000,000 rows, 57.2s elapsed
  Scanned 100 chunks, 100,000,000 rows, 71.7s elapsed
  Scanned 120 chunks, 120,000,000 rows, 85.7s elapsed
  Scanned 140 chunks, 140,000,000 rows, 99.5s elapsed
  Scanned 160 chunks, 160,000,000 rows, 112.6s elapsed
  Scanned 180 chunks, 180,000,000 rows, 125.8s elapsed
✓ Scanned 196,685,784 rows in 136.9s. Checking continuity...

Grid cells with any data: 11,211
Fully continuous (1h steps, no gaps): 11,211 (100.0%)
Cells with at least one gap: 0 (0.0%)

Gap size distribution (hours skipped, top 15):


In [1]:
# BUILD 6H-AHEAD FLOOD REGRESSION TARGET — MEMORY-SAFE VERSION
import os
import time
import numpy as np
import pandas as pd

CLEANED_DIR = "merged_output_hourly_imerg"
TARGET_PATH = os.path.join(CLEANED_DIR, "merged_data_final.csv")
TEMP_PATH = os.path.join(CLEANED_DIR, "merged_data_final.tmp.csv")
CHUNK_SIZE = 1_000_000

UK_BOUNDS = {"lat_min": 48.5, "lat_max": 62.4, "lon_min": -9.75, "lon_max": 3.25}
GRID_RES = 0.1
LEAD_HOURS = 6
ACCUM_WINDOW_HOURS = 24

PRECIP_COL = "imerg_precipitation"
SOIL_COLS = ["era5_swvl1", "era5_swvl2", "era5_swvl3"]
LSM_COL = "era5_lsm"
LAND_THRESHOLD = 0.5
PRECIP_WEIGHT = 0.6
SOIL_WEIGHT = 0.4
N_HIST_BINS = 1000

n_lat = int(round((UK_BOUNDS["lat_max"] - UK_BOUNDS["lat_min"]) / GRID_RES)) + 1
n_lon = int(round((UK_BOUNDS["lon_max"] - UK_BOUNDS["lon_min"]) / GRID_RES)) + 1
n_cells = n_lat * n_lon
print(f"Grid: {n_lat} lat x {n_lon} lon = {n_cells:,} cells")


def parse_hours(time_series):
    time_str = time_series.astype(str)
    date_only = time_str.str.len() == 10
    time_str = time_str.where(~date_only, time_str + "T00:00:00")
    time_str = time_str.str.replace(" ", "T", regex=False)
    parsed = pd.to_datetime(time_str, format="%Y-%m-%dT%H:%M:%S", errors="coerce")
    hours = (parsed - pd.Timestamp("1970-01-01")) // pd.Timedelta(hours=1)
    return hours.to_numpy(dtype=np.int64)


def cell_key(lat, lon):
    lat_idx = np.clip(
        np.round((lat.astype(np.float64) - UK_BOUNDS["lat_min"]) / GRID_RES).astype(np.int64),
        0, n_lat - 1
    )
    lon_idx = np.clip(
        np.round((lon.astype(np.float64) - UK_BOUNDS["lon_min"]) / GRID_RES).astype(np.int64),
        0, n_lon - 1
    )
    return lat_idx * n_lon + lon_idx


def histogram_percentile_rank(values, n_bins=N_HIST_BINS):
    """Approximate percentile rank via histogram.
    Returns float32 array in [0, 100]. O(n) memory vs O(n log n) for true rank."""
    finite = values[np.isfinite(values)]
    if len(finite) == 0:
        return np.full(len(values), np.nan, dtype=np.float32)
    lo, hi = float(finite.min()), float(finite.max())
    if lo == hi:
        return np.full(len(values), 50.0, dtype=np.float32)
    counts, edges = np.histogram(finite, bins=n_bins, range=(lo, hi))
    cumcounts = np.concatenate([[0], np.cumsum(counts)])
    total = cumcounts[-1]
    bin_indices = np.clip(np.searchsorted(edges[1:], values, side="left"), 0, n_bins - 1)
    return (100.0 * cumcounts[bin_indices] / total).astype(np.float32)


# PASS 1: global hour range
print("\nPass 1/3: finding global time range...")
t0 = time.time()
min_hour, max_hour = None, None
for chunk in pd.read_csv(TARGET_PATH, usecols=["time"], chunksize=CHUNK_SIZE,
                          on_bad_lines="skip"):
    hours = parse_hours(chunk["time"])
    cmin, cmax = hours.min(), hours.max()
    min_hour = cmin if min_hour is None else min(min_hour, cmin)
    max_hour = cmax if max_hour is None else max(max_hour, cmax)
n_hours = int(max_hour - min_hour + 1)
print(f"  {n_hours:,} hours ({time.time()-t0:.1f}s)")

# PASS 2: per-cell lookup arrays
print("\nPass 2/3: building lookup arrays...")
t0 = time.time()
precip_lookup = np.full((n_cells, n_hours), np.nan, dtype=np.float32)
soil_lookup = np.full((n_cells, n_hours), np.nan, dtype=np.float32)

usecols = ["time", "latitude", "longitude", PRECIP_COL] + SOIL_COLS
n_rows_seen = 0
for i, chunk in enumerate(pd.read_csv(TARGET_PATH, usecols=usecols,
                                        chunksize=CHUNK_SIZE, on_bad_lines="skip")):
    hours = parse_hours(chunk["time"]) - min_hour
    keys = cell_key(chunk["latitude"].to_numpy(), chunk["longitude"].to_numpy())
    precip_lookup[keys, hours] = chunk[PRECIP_COL].to_numpy(dtype=np.float32)
    soil_lookup[keys, hours] = chunk[SOIL_COLS].mean(axis=1).to_numpy(dtype=np.float32)
    n_rows_seen += len(chunk)
    if (i + 1) % 20 == 0:
        print(f"  ... {n_rows_seen:,} rows indexed, {time.time()-t0:.1f}s")

print(f"  Lookup built ({time.time()-t0:.1f}s)")

# Rolling accumulation — free intermediates immediately
print(f"\nComputing {ACCUM_WINDOW_HOURS}h rolling accumulation...")
precip_filled = np.nan_to_num(precip_lookup, nan=0.0)
del precip_lookup
cumsum = np.cumsum(precip_filled, axis=1)
del precip_filled
cumsum_padded = np.pad(cumsum, ((0,0),(ACCUM_WINDOW_HOURS,0)), constant_values=0)
del cumsum
accum_precip = cumsum_padded[:, ACCUM_WINDOW_HOURS:] - cumsum_padded[:, :-ACCUM_WINDOW_HOURS]
del cumsum_padded

# Composite score via histogram percentile ranking
print("Computing composite flood risk score...")
valid_mask = ~np.isnan(accum_precip) & ~np.isnan(soil_lookup)
precip_flat = accum_precip[valid_mask]
soil_flat = soil_lookup[valid_mask]
del accum_precip, soil_lookup

print(f"  Valid values to rank: {len(precip_flat):,}")
precip_pct = histogram_percentile_rank(precip_flat)
del precip_flat
soil_pct = histogram_percentile_rank(soil_flat)
del soil_flat

composite_flat = PRECIP_WEIGHT * precip_pct + SOIL_WEIGHT * soil_pct
del precip_pct, soil_pct

composite_score = np.full((n_cells, n_hours), np.nan, dtype=np.float32)
composite_score[valid_mask] = composite_flat
del composite_flat, valid_mask
print(f"  Score range: {np.nanmin(composite_score):.2f} - {np.nanmax(composite_score):.2f}")

# PASS 3: write output
print(f"\nPass 3/3: writing flood_composite_6h_ahead...")
t0 = time.time()
n_rows = 0
n_labeled = 0
first_chunk = True

for i, chunk in enumerate(pd.read_csv(TARGET_PATH, chunksize=CHUNK_SIZE,
                                        on_bad_lines="skip")):
    hours = parse_hours(chunk["time"]) - min_hour
    keys = cell_key(chunk["latitude"].to_numpy(), chunk["longitude"].to_numpy())
    future_hours = hours + LEAD_HOURS
    in_range = future_hours < n_hours

    future_score = np.full(len(chunk), np.nan, dtype=np.float32)
    future_score[in_range] = composite_score[keys[in_range], future_hours[in_range]]

    if "is_land" in chunk.columns:
        is_land = chunk["is_land"].to_numpy(dtype=bool)
    elif LSM_COL in chunk.columns:
        is_land = chunk[LSM_COL].to_numpy() > LAND_THRESHOLD
    else:
        is_land = np.ones(len(chunk), dtype=bool)

    valid = is_land & in_range & ~np.isnan(future_score)
    chunk["flood_composite_6h_ahead"] = np.where(valid, future_score, np.nan)
    n_labeled += int(valid.sum())

    chunk.to_csv(TEMP_PATH, mode="w" if first_chunk else "a",
                 header=first_chunk, index=False)
    first_chunk = False
    n_rows += len(chunk)
    if (i + 1) % 20 == 0:
        print(f"  ... {n_rows:,} rows written, {time.time()-t0:.1f}s")

print(f"\n  Wrote {n_rows:,} rows in {time.time()-t0:.1f}s")
print(f"  Labeled (land, in range): {n_labeled:,} ({100*n_labeled/n_rows:.2f}%)")
os.replace(TEMP_PATH, TARGET_PATH)
print(f"Replaced {TARGET_PATH} in place — added flood_composite_6h_ahead")

Grid: 140 lat x 131 lon = 18,340 cells

Pass 1/3: finding global time range...
  21,888 hours (303.6s)

Pass 2/3: building lookup arrays...
  ... 20,000,000 rows indexed, 20.6s
  ... 40,000,000 rows indexed, 40.0s
  ... 60,000,000 rows indexed, 59.4s
  ... 80,000,000 rows indexed, 78.8s
  ... 100,000,000 rows indexed, 98.1s
  ... 120,000,000 rows indexed, 117.4s
  ... 140,000,000 rows indexed, 136.5s
  ... 160,000,000 rows indexed, 155.9s
  ... 180,000,000 rows indexed, 175.0s
  ... 200,000,000 rows indexed, 194.1s
  ... 220,000,000 rows indexed, 213.0s
  ... 240,000,000 rows indexed, 232.1s
  ... 260,000,000 rows indexed, 251.4s
  ... 280,000,000 rows indexed, 270.5s
  ... 300,000,000 rows indexed, 289.9s
  ... 320,000,000 rows indexed, 309.3s
  ... 340,000,000 rows indexed, 328.5s
  ... 360,000,000 rows indexed, 348.3s
  ... 380,000,000 rows indexed, 367.3s
  ... 400,000,000 rows indexed, 386.3s
  Lookup built (387.7s)

Computing 24h rolling accumulation...
Computing composite flood 

## Correlation Heatmap

In [11]:
# CORRELATION HEATMAP — LAND CELLS ONLY, RELEVANT FEATURES ONLY
# ============================================================================
# Excludes: lat, lon, time (coordinates), era5_sst and era5_skt
# (sea-only / mixed-meaning variables not relevant for flood prediction
# on land), and is_land itself (binary mask, not a predictive feature).
#
# Samples land-only rows using vectorised numpy random choice per chunk —
# replaces the iterrows() approach which was causing ~4 hour runtimes.
# ============================================================================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CSV_PATH = "merged_output_hourly_imerg/merged_data_final.csv"
OUTPUT_DIR = "eda_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Columns to exclude from correlation
EXCLUDE_COLS = {
    "time", "latitude", "longitude",
    "era5_sst",
    "era5_skt",
    "is_land",
    "flood_composite_6h_ahead",   # target variable — separate analysis
}

# Land mask column — used to filter rows, then excluded from correlation
LAND_COL = "era5_lsm"    # fallback if is_land not present
LAND_THRESHOLD = 0.5

SAMPLE_SIZE = 250_000
CHUNK_SIZE = 500_000
SEED = 42
rng = np.random.default_rng(SEED)

print(f"Source: {CSV_PATH} ({os.path.getsize(CSV_PATH)/1e9:.2f} GB)")

all_cols = pd.read_csv(CSV_PATH, nrows=0).columns.tolist()

# Determine land column
if "is_land" in all_cols:
    land_filter_col = "is_land"
elif LAND_COL in all_cols:
    land_filter_col = LAND_COL
else:
    land_filter_col = None
    print("⚠ No land column found — using all rows")

corr_cols = [c for c in all_cols
             if c not in EXCLUDE_COLS
             and c != land_filter_col
             and pd.api.types.is_numeric_dtype(
                 pd.read_csv(CSV_PATH, nrows=2)[c] if c in all_cols else pd.Series()
             )]

# Faster dtype check using the header sample we already have
header_sample = pd.read_csv(CSV_PATH, nrows=2)
corr_cols = [c for c in all_cols
             if c not in EXCLUDE_COLS
             and c != land_filter_col
             and c in header_sample.columns
             and pd.api.types.is_numeric_dtype(header_sample[c])]

print(f"Correlation columns ({len(corr_cols)}): {corr_cols}")
print(f"Land filter: '{land_filter_col}' > {LAND_THRESHOLD}")
print(f"Sampling {SAMPLE_SIZE:,} land rows using vectorised numpy...\n")

# Vectorised reservoir sampling — collect all land rows from each chunk,
# then subsample. Far faster than row-by-row iterrows().
collected = []
n_land_seen = 0

read_cols = corr_cols + ([land_filter_col] if land_filter_col else [])
for i, chunk in enumerate(pd.read_csv(CSV_PATH, usecols=read_cols,
                                        chunksize=CHUNK_SIZE, on_bad_lines="skip")):
    if land_filter_col:
        if land_filter_col == "is_land":
            land_mask = chunk[land_filter_col] == 1
        else:
            land_mask = chunk[land_filter_col] > LAND_THRESHOLD
        land_chunk = chunk.loc[land_mask, corr_cols]
    else:
        land_chunk = chunk[corr_cols]

    if len(land_chunk) == 0:
        continue

    n_land_seen += len(land_chunk)
    collected.append(land_chunk)

    if (i + 1) % 40 == 0:
        print(f"  ... {n_land_seen:,} land rows seen so far")

print(f"✓ Scanned file: {n_land_seen:,} land rows found")

# Concatenate and subsample
all_land = pd.concat(collected, ignore_index=True)
del collected

if len(all_land) > SAMPLE_SIZE:
    sample_df = all_land.sample(n=SAMPLE_SIZE, random_state=SEED)
else:
    sample_df = all_land
    print(f"⚠ Fewer land rows than sample size — using all {len(sample_df):,}")

del all_land
print(f"✓ Sample size: {len(sample_df):,} rows, {len(corr_cols)} columns\n")

# Compute correlation
print("Computing Pearson correlation matrix...")
corr = sample_df.corr(method="pearson", numeric_only=True)
n = len(corr)

# Save CSV
corr.to_csv(os.path.join(OUTPUT_DIR, "correlation_matrix_land_only.csv"),
             float_format="%.4f")
print(f"✓ Saved correlation_matrix_land_only.csv")

# Plot heatmap
fig, ax = plt.subplots(figsize=(max(10, n * 0.75), max(8, n * 0.65)))
cmap = plt.cm.RdBu_r
im = ax.imshow(corr.values, cmap=cmap, vmin=-1, vmax=1, aspect="auto")
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="Pearson r")

ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(corr.columns, rotation=45, ha="right", fontsize=8)
ax.set_yticklabels(corr.index, fontsize=8)

for i in range(n):
    for j in range(n):
        val = corr.values[i, j]
        if not np.isnan(val):
            text_color = "white" if abs(val) > 0.6 else "black"
            ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                    fontsize=6 if n > 15 else 8, color=text_color)

ax.set_title(
    f"Pearson correlation — land cells only\n"
    f"(reservoir sample n={len(sample_df):,}, excluding sea-only variables)",
    fontsize=12, pad=14
)
fig.tight_layout()
heatmap_path = os.path.join(OUTPUT_DIR, "correlation_heatmap_land_only.png")
fig.savefig(heatmap_path, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"✓ Saved {heatmap_path}")

# Print top correlated pairs
pairs = []
cols_list = corr.columns.tolist()
for i in range(len(cols_list)):
    for j in range(i + 1, len(cols_list)):
        val = corr.values[i, j]
        if not np.isnan(val):
            pairs.append((abs(val), val, cols_list[i], cols_list[j]))
pairs.sort(reverse=True)

print(f"\nTop 20 correlated pairs (land cells only):")
for _, r, c1, c2 in pairs[:20]:
    print(f"  r={r:+.3f}  {c1}  <->  {c2}")

print(f"\n✓ Done. Outputs in: {OUTPUT_DIR}/")

Source: merged_output_hourly_imerg/merged_data_final.csv (70.31 GB)
Correlation columns (17): ['era5_d2m', 'era5_msl', 'era5_swvl1', 'era5_z', 'era5_cape', 'era5_tcwv', 'era5_blh', 'era5_cp', 'era5_lsm', 'imerg_precipitation', 'imerg_precipitationQualityIndex', 'imerg_probabilityLiquidPrecipitation', 'wind_speed10', 'soil_no_flood_soil', 'soil_average_soil', 'soil_flood_soil', 'soil_heavy_flood_soil']
Land filter: 'is_land' > 0.5
Sampling 250,000 land rows using vectorised numpy...



ParserError: Error tokenizing data. C error: Calling read(nbytes) on source failed. Try engine='python'.

In [1]:
# DROP HIGHLY CORRELATED COLUMNS (|r| > 0.70)
# ============================================================================
# Drops 4 columns identified as redundant from the land-only correlation
# heatmap, keeping the more physically meaningful member of each correlated
# group:
#
#   era5_swvl2  (r=0.98 with swvl1, r=0.96 with swvl3) -> keep swvl1
#   era5_swvl3  (r=0.92 with swvl1, r=0.96 with swvl2) -> keep swvl1
#   era5_skt    (r=0.88 with d2m,   r=0.72 with tcwv)  -> keep d2m + tcwv
#   era5_sp     (r=0.74 with msl,   r=-0.72 with z)    -> keep msl
#
# Remaining feature set: 15 input columns (down from 19)
# ============================================================================
import os
import pandas as pd

TARGET_PATH = "merged_output_hourly_imerg/merged_data_final.csv"
TEMP_PATH = "merged_output_hourly_imerg/merged_data_final.tmp.csv"
CHUNK_SIZE = 1_000_000

COLS_TO_DROP = ["era5_swvl2", "era5_swvl3", "era5_skt", "era5_sp"]

# Pre-flight check
header_cols = pd.read_csv(TARGET_PATH, nrows=0).columns.tolist()
actually_present = [c for c in COLS_TO_DROP if c in header_cols]
already_gone = [c for c in COLS_TO_DROP if c not in header_cols]

if already_gone:
    print(f"Note: already absent — {already_gone}")
if not actually_present:
    print("✓ All target columns already removed — nothing to do.")
else:
    print(f"Dropping: {actually_present}")
    keep_cols = [c for c in header_cols if c not in COLS_TO_DROP]
    print(f"Keeping {len(keep_cols)} of {len(header_cols)} columns")
    print(f"Source: {TARGET_PATH} ({os.path.getsize(TARGET_PATH)/1e9:.2f} GB)\n")

    n_rows = 0
    first_chunk = True
    for i, chunk in enumerate(pd.read_csv(TARGET_PATH, chunksize=CHUNK_SIZE,
                                            on_bad_lines="skip")):
        chunk = chunk.drop(columns=actually_present, errors="ignore")
        chunk.to_csv(TEMP_PATH, mode="w" if first_chunk else "a",
                     header=first_chunk, index=False)
        first_chunk = False
        n_rows += len(chunk)
        if (i + 1) % 20 == 0:
            print(f"  ... {n_rows:,} rows written")

    print(f"\n✓ Wrote {n_rows:,} rows")

    # Verify before replacing
    out_cols = pd.read_csv(TEMP_PATH, nrows=0).columns.tolist()
    still_present = [c for c in COLS_TO_DROP if c in out_cols]
    if still_present:
        print(f"⚠ Columns still present in output: {still_present} — NOT replacing.")
        os.remove(TEMP_PATH)
    else:
        os.replace(TEMP_PATH, TARGET_PATH)
        print(f"✓ Replaced {TARGET_PATH} in place")
        print(f"  Dropped: {actually_present}")
        print(f"  Remaining columns ({len(out_cols)}): {out_cols}")
        print(f"  New file size: {os.path.getsize(TARGET_PATH)/1e9:.2f} GB")

Note: already absent — ['era5_swvl2', 'era5_swvl3', 'era5_sp']
Dropping: ['era5_skt']
Keeping 23 of 24 columns
Source: merged_output_hourly_imerg/merged_data_final.csv (70.31 GB)

  ... 20,000,000 rows written
  ... 40,000,000 rows written
  ... 60,000,000 rows written
  ... 80,000,000 rows written
  ... 100,000,000 rows written
  ... 120,000,000 rows written
  ... 140,000,000 rows written
  ... 160,000,000 rows written
  ... 180,000,000 rows written
  ... 200,000,000 rows written
  ... 220,000,000 rows written
  ... 240,000,000 rows written
  ... 260,000,000 rows written
  ... 280,000,000 rows written
  ... 300,000,000 rows written
  ... 320,000,000 rows written
  ... 340,000,000 rows written
  ... 360,000,000 rows written
  ... 380,000,000 rows written
  ... 400,000,000 rows written

✓ Wrote 401,425,920 rows
✓ Replaced merged_output_hourly_imerg/merged_data_final.csv in place
  Dropped: ['era5_skt']
  Remaining columns (23): ['time', 'latitude', 'longitude', 'era5_d2m', 'era5_msl', '

In [6]:
# DISTRIBUTION HISTOGRAMS — RAW (PRE-NORMALISATION) DATA
# ============================================================================
# Reads merged_data_final.csv (unnormalised) and generates histograms for
# all numeric attributes. Column list is read dynamically from the file
# header so it stays current regardless of which columns have been added
# or dropped. Bin ranges are computed from the actual data in one pass.
# ============================================================================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CLEANED_DIR = "merged_output_hourly_imerg"
CSV_PATH = os.path.join(CLEANED_DIR, "merged_data_final.csv")
OUTPUT_DIR = "eda_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

CHUNK_SIZE = 500_000
HIST_BINS = 60
EXCLUDE_COLS = {"time", "date", "latitude", "longitude"}

# Read columns dynamically
all_cols = pd.read_csv(CSV_PATH, nrows=0).columns.tolist()
hist_cols = [c for c in all_cols
             if c not in EXCLUDE_COLS
             and pd.api.types.is_numeric_dtype(
                 pd.read_csv(CSV_PATH, nrows=2)[c])]

print(f"Source: {CSV_PATH} ({os.path.getsize(CSV_PATH)/1e9:.2f} GB)")
print(f"Generating histograms for {len(hist_cols)} columns:")
print(f"  {hist_cols}\n")

# Pass 1: compute actual min/max per column for bin edges
print("Pass 1/2: scanning min/max for bin edges...")
actual_min = {col: np.inf  for col in hist_cols}
actual_max = {col: -np.inf for col in hist_cols}

for chunk in pd.read_csv(CSV_PATH, usecols=hist_cols, chunksize=CHUNK_SIZE,
                          on_bad_lines="skip"):
    for col in hist_cols:
        v = pd.to_numeric(chunk[col], errors="coerce").dropna()
        if len(v) == 0:
            continue
        actual_min[col] = min(actual_min[col], float(v.min()))
        actual_max[col] = max(actual_max[col], float(v.max()))

bin_edges = {}
hist_counts = {}
for col in hist_cols:
    lo, hi = actual_min[col], actual_max[col]
    if not np.isfinite(lo) or not np.isfinite(hi) or lo == hi:
        bin_edges[col] = None
        continue
    bin_edges[col] = np.linspace(lo, hi, HIST_BINS + 1)
    hist_counts[col] = np.zeros(HIST_BINS, dtype=np.int64)

# Pass 2: accumulate histogram counts
print("Pass 2/2: accumulating histogram counts...")
n_rows = 0
col_means = {col: 0.0 for col in hist_cols}
col_counts = {col: 0   for col in hist_cols}

reader = pd.read_csv(CSV_PATH, usecols=hist_cols, chunksize=CHUNK_SIZE,
                     on_bad_lines="skip")
for i, chunk in enumerate(reader):
    n_rows += len(chunk)
    for col in hist_cols:
        if bin_edges[col] is None:
            continue
        vals = pd.to_numeric(chunk[col], errors="coerce").dropna().to_numpy()
        if vals.size == 0:
            continue
        counts, _ = np.histogram(vals, bins=bin_edges[col])
        hist_counts[col] += counts
        col_means[col]  += vals.sum()
        col_counts[col] += len(vals)

    if (i + 1) % 40 == 0:
        print(f"  ... {n_rows:,} rows processed")

print(f"✓ Processed {n_rows:,} rows")

# Finalise means
for col in hist_cols:
    if col_counts[col] > 0:
        col_means[col] /= col_counts[col]

# Plot
n_cols_grid = 4
n_rows_grid = int(np.ceil(len(hist_cols) / n_cols_grid))
fig, axes = plt.subplots(n_rows_grid, n_cols_grid,
                          figsize=(4.2 * n_cols_grid, 3.2 * n_rows_grid))
axes = np.array(axes).reshape(-1)

for idx, col in enumerate(hist_cols):
    ax = axes[idx]
    if bin_edges[col] is None:
        ax.set_title(f"{col}\n(constant or invalid range)", fontsize=10)
        ax.axis("off")
        continue
    edges = bin_edges[col]
    centers = (edges[:-1] + edges[1:]) / 2
    width = edges[1] - edges[0]
    ax.bar(centers, hist_counts[col], width=width,
           color="steelblue", edgecolor="none")
    mean_val = col_means[col]
    if np.isfinite(mean_val):
        ax.axvline(mean_val, color="orange", linestyle="--", linewidth=1.5)
    ax.set_title(col, fontsize=10)
    ax.tick_params(labelsize=8)

for idx in range(len(hist_cols), len(axes)):
    axes[idx].axis("off")

fig.suptitle("Distribution of each attribute — RAW (pre-normalisation)\n"
             "(orange dashed = mean)", fontsize=13)
fig.tight_layout(rect=[0, 0, 1, 0.96])

hist_path = os.path.join(OUTPUT_DIR, "distributions_grid_raw.png")
fig.savefig(hist_path, dpi=140, bbox_inches="tight")
plt.close(fig)
print(f"✓ Saved -> {hist_path}")

Source: merged_output_hourly_imerg/merged_data_final.csv (70.31 GB)
Generating histograms for 21 columns:
  ['era5_d2m', 'era5_msl', 'era5_sst', 'era5_swvl1', 'era5_z', 'era5_cape', 'era5_tcwv', 'era5_blh', 'era5_cp', 'era5_lsm', 'imerg_precipitation', 'imerg_precipitationQualityIndex', 'imerg_probabilityLiquidPrecipitation', 'is_land', 'wind_speed10', 'flood_composite_6h_ahead', 'soil_no_flood_soil', 'soil_average_soil', 'soil_flood_soil', 'soil_heavy_flood_soil', 'era5_t2m']

Pass 1/2: scanning min/max for bin edges...
Pass 2/2: accumulating histogram counts...
  ... 20,000,000 rows processed
  ... 40,000,000 rows processed
  ... 60,000,000 rows processed
  ... 80,000,000 rows processed
  ... 100,000,000 rows processed
  ... 120,000,000 rows processed
  ... 140,000,000 rows processed
  ... 160,000,000 rows processed
  ... 180,000,000 rows processed
  ... 200,000,000 rows processed
  ... 220,000,000 rows processed
  ... 240,000,000 rows processed
  ... 260,000,000 rows processed
  ...

In [6]:
# REMAP era5_slt TO FLOOD-RISK CATEGORIES + ONE-HOT ENCODE
# ============================================================================
# Maps the 7 FAO soil type integer codes to 4 physically-meaningful flood
# risk categories, then one-hot encodes them into 4 binary columns.
# Drops the original era5_slt integer column and imerg_MWprecipSource
# (redundant with imerg_precipitationQualityIndex).
#
# Mapping:
#   0, 7, 8 (unknown/tropical/artificial) -> average_soil (fallback)
#   1 (coarse/sandy)                       -> no_flood_soil
#   2 (medium/loam), 3 (medium fine)       -> average_soil
#   4 (fine/clay)                           -> flood_soil
#   5 (very fine/heavy clay)               -> heavy_flood_soil
#   6 (organic/peat)                        -> flood_soil
# ============================================================================
import os
import numpy as np
import pandas as pd

TARGET_PATH = "merged_output_hourly_imerg/merged_data_final.csv"
TEMP_PATH = "merged_output_hourly_imerg/merged_data_final.tmp.csv"
CHUNK_SIZE = 1_000_000

SOIL_MAP = {
    0: "average_soil",       # no data — fallback
    1: "no_flood_soil",      # coarse/sandy — high infiltration
    2: "average_soil",       # medium/loam
    3: "average_soil",       # medium fine
    4: "flood_soil",         # fine/clay — low infiltration
    5: "heavy_flood_soil",   # very fine/heavy clay — very low infiltration
    6: "flood_soil",         # organic/peat — frequently saturated in UK
    7: "average_soil",       # tropical organic — fallback
    8: "average_soil",       # artificial/no data — fallback
}

# All possible categories in a fixed order — ensures consistent one-hot
# column ordering across all chunks regardless of which codes appear
CATEGORIES = ["no_flood_soil", "average_soil", "flood_soil", "heavy_flood_soil"]
OHE_COLS = [f"soil_{c}" for c in CATEGORIES]

DROP_COLS = ["era5_slt", "imerg_MWprecipSource"]

# Pre-flight check
header_cols = pd.read_csv(TARGET_PATH, nrows=0).columns.tolist()
missing = [c for c in ["era5_slt"] if c not in header_cols]
if missing:
    print(f"⚠ era5_slt not found in file — check column names: {header_cols}")
else:
    print(f"Source: {TARGET_PATH} ({os.path.getsize(TARGET_PATH)/1e9:.2f} GB)")
    print(f"Remapping era5_slt -> {OHE_COLS}")
    print(f"Dropping: {[c for c in DROP_COLS if c in header_cols]}\n")

    n_rows = 0
    first_chunk = True
    category_counts = {c: 0 for c in CATEGORIES}

    for i, chunk in enumerate(pd.read_csv(TARGET_PATH, chunksize=CHUNK_SIZE,
                                            on_bad_lines="skip")):
        # Map integer soil code to category string
        slt_raw = chunk["era5_slt"].fillna(0).astype(int)
        soil_cat = slt_raw.map(SOIL_MAP).fillna("average_soil")

        # One-hot encode into 4 fixed binary columns
        for cat, col in zip(CATEGORIES, OHE_COLS):
            chunk[col] = (soil_cat == cat).astype(np.int8)
            category_counts[cat] += int((soil_cat == cat).sum())

        # Drop original integer column and MWprecipSource
        chunk = chunk.drop(columns=[c for c in DROP_COLS if c in chunk.columns],
                           errors="ignore")

        chunk.to_csv(TEMP_PATH, mode="w" if first_chunk else "a",
                     header=first_chunk, index=False)
        first_chunk = False
        n_rows += len(chunk)
        if (i + 1) % 20 == 0:
            print(f"  ... {n_rows:,} rows written")

    print(f"\n✓ Wrote {n_rows:,} rows")
    print(f"\nSoil category distribution:")
    for cat, col in zip(CATEGORIES, OHE_COLS):
        pct = 100 * category_counts[cat] / n_rows
        print(f"  {col:<25} {category_counts[cat]:>15,}  ({pct:.2f}%)")

    # Verify
    out_cols = pd.read_csv(TEMP_PATH, nrows=0).columns.tolist()
    ohe_present = all(c in out_cols for c in OHE_COLS)
    slt_gone = "era5_slt" not in out_cols

    if ohe_present and slt_gone:
        os.replace(TEMP_PATH, TARGET_PATH)
        print(f"\n✓ Replaced {TARGET_PATH} in place")
        print(f"  Added: {OHE_COLS}")
        print(f"  Dropped: {[c for c in DROP_COLS if c not in out_cols]}")
        print(f"  Final columns ({len(out_cols)}): {out_cols}")
    else:
        print(f"\n⚠ Verification failed — NOT replacing source.")
        if not ohe_present:
            print(f"  Missing OHE columns: {[c for c in OHE_COLS if c not in out_cols]}")
        if not slt_gone:
            print(f"  era5_slt still present")
        os.remove(TEMP_PATH)

Source: merged_output_hourly_imerg/merged_data_final.csv (72.23 GB)
Remapping era5_slt -> ['soil_no_flood_soil', 'soil_average_soil', 'soil_flood_soil', 'soil_heavy_flood_soil']
Dropping: ['era5_slt', 'imerg_MWprecipSource']

  ... 20,000,000 rows written
  ... 40,000,000 rows written
  ... 60,000,000 rows written
  ... 80,000,000 rows written
  ... 100,000,000 rows written
  ... 120,000,000 rows written
  ... 140,000,000 rows written
  ... 160,000,000 rows written
  ... 180,000,000 rows written
  ... 200,000,000 rows written
  ... 220,000,000 rows written
  ... 240,000,000 rows written
  ... 260,000,000 rows written
  ... 280,000,000 rows written
  ... 300,000,000 rows written
  ... 320,000,000 rows written
  ... 340,000,000 rows written
  ... 360,000,000 rows written
  ... 380,000,000 rows written
  ... 400,000,000 rows written

✓ Wrote 401,425,920 rows

Soil category distribution:
  soil_no_flood_soil             43,491,456  (10.83%)
  soil_average_soil             346,924,800  (86.

In [8]:
# FILL era5_sst NaN (LAND CELLS) WITH MINIMUM SEA SST
# ============================================================================
# ERA5 SST is NaN over land by definition. Following GenCast (Google
# DeepMind, 2024): "values over land are replaced with the minimum sea
# surface temperature seen globally in a subset of ERA5." This is more
# defensible than mean-filling because:
#   - The minimum is a real, physically-observed value (not invented)
#   - It sits far enough below typical SST values that the model can
#     easily distinguish fill values on land cells from real cold sea
#     temperatures, especially when combined with the is_land column
#
# Applied to BOTH files:
#   merged_data_final.csv      (unnormalised source)
#   merged_data_normalised.csv (normalised model input)
# ============================================================================
import os
import numpy as np
import pandas as pd

CLEANED_DIR = "merged_output_hourly_imerg"
FILES = {
    "unnormalised": os.path.join(CLEANED_DIR, "merged_data_final.csv"),
    "normalised":   os.path.join(CLEANED_DIR, "merged_data_normalised.csv"),
}
CHUNK_SIZE = 1_000_000
SST_COL = "era5_sst"
LAND_COL = "era5_lsm"
LAND_THRESHOLD = 0.5

for label, path in FILES.items():
    if not os.path.exists(path):
        print(f"⚠ {path} not found — skipping")
        continue

    temp_path = path.replace(".csv", ".tmp.csv")
    header_cols = pd.read_csv(path, nrows=0).columns.tolist()

    if SST_COL not in header_cols:
        print(f"⚠ '{SST_COL}' not found in {label} file — skipping")
        continue

    print(f"\n{'='*70}")
    print(f"Processing: {label}")
    print(f"File: {path}")
    print(f"{'='*70}")

    # ── Pass 1: find global minimum SST from sea cells ──
    print("Pass 1/2: finding global minimum SST from sea cells...")
    sst_min = np.inf
    sst_count = 0
    n_nan_land = 0

    for chunk in pd.read_csv(path, usecols=[SST_COL, LAND_COL],
                              chunksize=CHUNK_SIZE, on_bad_lines="skip"):
        is_sea = chunk[LAND_COL] <= LAND_THRESHOLD
        sea_sst = chunk.loc[is_sea, SST_COL].dropna()

        if len(sea_sst) > 0:
            sst_count += len(sea_sst)
            chunk_min = float(sea_sst.min())
            if chunk_min < sst_min:
                sst_min = chunk_min  # track global minimum across ALL chunks

        n_nan_land += int(chunk.loc[~is_sea, SST_COL].isna().sum())

    if sst_count == 0 or not np.isfinite(sst_min):
        print(f"⚠ No valid sea SST values found — skipping {label}")
        continue

    fill_value = sst_min
    print(f"  Sea cells with valid SST: {sst_count:,}")
    print(f"  Global minimum SST (fill value): {fill_value:.4f}")
    print(f"  Land cells with NaN SST to fill: {n_nan_land:,}")

    # ── Pass 2: fill NaN SST on land cells ──
    print("Pass 2/2: filling land-cell NaN SST values...")
    n_rows = 0
    n_filled = 0
    first_chunk = True

    for chunk in pd.read_csv(path, chunksize=CHUNK_SIZE, on_bad_lines="skip"):
        nan_mask = chunk[SST_COL].isna()
        n_filled += int(nan_mask.sum())
        chunk[SST_COL] = chunk[SST_COL].fillna(fill_value).astype(np.float32)

        chunk.to_csv(temp_path, mode="w" if first_chunk else "a",
                     header=first_chunk, index=False)
        first_chunk = False
        n_rows += len(chunk)

    print(f"  Rows processed: {n_rows:,}")
    print(f"  NaN values filled: {n_filled:,}")

    # ── Verify ──
    sample = pd.read_csv(temp_path, usecols=[SST_COL], nrows=18340)
    remaining_nan = sample[SST_COL].isna().sum()
    if remaining_nan > 0:
        print(f"⚠ {remaining_nan} NaN values still present — NOT replacing.")
        os.remove(temp_path)
    else:
        os.replace(temp_path, path)
        print(f"✓ Replaced {path} in place")
        print(f"  era5_sst range in sample: "
              f"{sample[SST_COL].min():.4f} - {sample[SST_COL].max():.4f}")
        print(f"  (filled land cells will show {fill_value:.4f}, "
              f"real sea values will be higher)")

print("\n✓ Done. era5_sst NaN values filled in both files.")
print("  Method: global minimum sea SST (following GenCast preprocessing)")


Processing: unnormalised
File: merged_output_hourly_imerg/merged_data_final.csv
Pass 1/2: finding global minimum SST from sea cells...
  Sea cells with valid SST: 290,716,416
  Global minimum SST (fill value): 277.2192
  Land cells with NaN SST to fill: 0
Pass 2/2: filling land-cell NaN SST values...


In [2]:
# FILL imerg_precipitation NaN — FAST VECTORISED VERSION WITH PROGRESS BAR
# ============================================================================
# Replaces the scipy generic_filter (Python callback, ~3-6 hours) with a
# fully vectorised np.roll-based 4-connected neighbour fill (~15-25 min).
# Applies to merged_data_final.csv only — merged_data_normalised.csv will
# be regenerated fresh from normalisation after this step.
# ============================================================================
import os
import time
import numpy as np
import pandas as pd
from tqdm import tqdm

CLEANED_DIR = "merged_output_hourly_imerg"
CHUNK_SIZE  = 18340  # exactly one hour — safe given confirmed gap-free grid

FILL_COLS = [
    "imerg_precipitation",
    "imerg_precipitationQualityIndex",
    "imerg_probabilityLiquidPrecipitation",
]

N_LAT   = 140
N_LON   = 131
N_HOURS = 21888

FILES = {
    "unnormalised": os.path.join(CLEANED_DIR, "merged_data_final.csv"),
    # normalised file excluded — will be regenerated from scratch after this
}


def fill_hour_grid(arr_2d):
    """Vectorised 4-connected neighbour fill using np.roll.
    ~100x faster than scipy generic_filter with Python callback.
    Iterates up to 5 times to handle clusters of adjacent NaN cells."""
    result = arr_2d.copy()
    for _ in range(5):
        nan_mask = np.isnan(result)
        if not nan_mask.any():
            break
        north = np.roll(result, -1, axis=0)
        south = np.roll(result,  1, axis=0)
        east  = np.roll(result, -1, axis=1)
        west  = np.roll(result,  1, axis=1)
        stack = np.stack([north, south, east, west], axis=0)
        with np.errstate(all="ignore"):
            neighbour_mean = np.nanmean(stack, axis=0)
        result = np.where(nan_mask, neighbour_mean, result)
    return result


for label, path in FILES.items():
    if not os.path.exists(path):
        print(f"⚠ {path} not found — skipping")
        continue

    temp_path = path.replace(".csv", ".tmp.csv")
    header_cols = pd.read_csv(path, nrows=0).columns.tolist()
    active_fill_cols = [c for c in FILL_COLS if c in header_cols]

    if not active_fill_cols:
        print(f"⚠ No IMERG columns found in {label} — skipping")
        continue

    print(f"\n{'='*70}")
    print(f"Processing: {label}")
    print(f"File: {path} ({os.path.getsize(path)/1e9:.2f} GB)")
    print(f"Filling: {active_fill_cols}")
    print(f"{'='*70}\n")

    t0 = time.time()
    n_hours = 0
    n_cells_filled = {col: 0 for col in active_fill_cols}
    first_chunk = True

    reader = pd.read_csv(path, chunksize=CHUNK_SIZE, on_bad_lines="skip")

    with tqdm(total=N_HOURS, desc=f"Filling {label}", unit="hr",
              bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} hrs "
                         "[{elapsed}<{remaining}, {rate_fmt}]") as pbar:

        for chunk in reader:
            if len(chunk) != CHUNK_SIZE:
                # Final partial chunk — write as-is, no fill needed
                chunk.to_csv(temp_path, mode="a", header=False, index=False)
                continue

            for col in active_fill_cols:
                col_vals = chunk[col].to_numpy(dtype=np.float64)
                n_nan_before = int(np.isnan(col_vals).sum())

                if n_nan_before > 0:
                    grid_2d   = col_vals.reshape(N_LAT, N_LON)
                    filled_2d = fill_hour_grid(grid_2d)
                    chunk[col] = filled_2d.ravel().astype(np.float32)
                    n_cells_filled[col] += n_nan_before - int(
                        np.isnan(filled_2d).sum()
                    )

            chunk.to_csv(temp_path, mode="w" if first_chunk else "a",
                         header=first_chunk, index=False)
            first_chunk = False
            n_hours += 1
            pbar.update(1)

            # Update postfix with fill counts every 500 hours
            if n_hours % 500 == 0:
                pbar.set_postfix({
                    col.replace("imerg_", ""): f"{n_cells_filled[col]:,}"
                    for col in active_fill_cols
                })

    elapsed = time.time() - t0
    print(f"\n✓ Processed {n_hours:,} hours in {elapsed/60:.1f} min")
    for col in active_fill_cols:
        print(f"  {col}: {n_cells_filled[col]:,} NaN cells filled")

    # Verify
    print("\nVerifying NaN rates in first hour of output...")
    sample = pd.read_csv(temp_path, usecols=active_fill_cols, nrows=CHUNK_SIZE)
    all_clean = True
    for col in active_fill_cols:
        remaining = int(sample[col].isna().sum())
        status = "✓ clean" if remaining == 0 else f"⚠ {remaining} NaN remain"
        print(f"  {col}: {status}")
        if remaining > 0:
            all_clean = False

    if all_clean:
        os.replace(temp_path, path)
        print(f"\n✓ Replaced {path} in place")
        print(f"  New file size: {os.path.getsize(path)/1e9:.2f} GB")
    else:
        print(f"\n⚠ NaN values remain — original file kept, "
              f"temp file at {temp_path}")

print("\n✓ Done. Next step: re-run normalise_features_v2.py to regenerate "
      "merged_data_normalised.csv from the now fully-clean source.")


Processing: unnormalised
File: merged_output_hourly_imerg/merged_data_final.csv (70.30 GB)
Filling: ['imerg_precipitation', 'imerg_precipitationQualityIndex', 'imerg_probabilityLiquidPrecipitation']



Filling unnormalised:  17%|█▋        | 3648/21888 hrs [09:00<44:28,  6.84hr/s]  /var/folders/f9/b36b9whj67b0j3195q6474dh0000gn/T/ipykernel_12051/3839038586.py:48: RuntimeWarning: Mean of empty slice
  neighbour_mean = np.nanmean(stack, axis=0)
Filling unnormalised:  17%|█▋        | 3649/21888 hrs [09:00<44:41,  6.80hr/s]/var/folders/f9/b36b9whj67b0j3195q6474dh0000gn/T/ipykernel_12051/3839038586.py:48: RuntimeWarning: Mean of empty slice
  neighbour_mean = np.nanmean(stack, axis=0)
Filling unnormalised: 100%|██████████| 21888/21888 hrs [53:10<00:00,  6.86hr/s]



✓ Processed 21,888 hours in 53.2 min
  imerg_precipitation: 45,638,314 NaN cells filled
  imerg_precipitationQualityIndex: 45,638,314 NaN cells filled
  imerg_probabilityLiquidPrecipitation: 45,636,480 NaN cells filled

Verifying NaN rates in first hour of output...
  imerg_precipitation: ✓ clean
  imerg_precipitationQualityIndex: ✓ clean
  imerg_probabilityLiquidPrecipitation: ✓ clean

✓ Replaced merged_output_hourly_imerg/merged_data_final.csv in place
  New file size: 71.11 GB

✓ Done. Next step: re-run normalise_features_v2.py to regenerate merged_data_normalised.csv from the now fully-clean source.


# NORMALISATION

In [6]:
# NORMALISE FEATURES v2 
# ============================================================================
# Updated from normalise_features_separate.py:
#   - era5_t2m added to ZSCORE_COLS (same treatment as era5_d2m)
#   - era5_skt dropped in the same pass (not written to output)
#   - Writes to merged_data_normalised.csv (separate file, source unchanged)
#   - Recomputes and saves normalisation_stats.csv with t2m included
# ============================================================================
import os
import numpy as np
import pandas as pd

SOURCE_PATH = "merged_output_hourly_imerg/merged_data_final.csv"
OUTPUT_PATH = "merged_output_hourly_imerg/merged_data_normalised.csv"
STATS_PATH  = "eda_output/normalisation_stats.csv"
CHUNK_SIZE  = 1_000_000

ZSCORE_COLS = [
    "era5_sst", "era5_d2m", "era5_t2m", "era5_msl", "era5_tcwv", "era5_blh", "wind_speed10"
]
LOG1P_ZSCORE_COLS = [
    "imerg_precipitation", "era5_cape", "era5_cp", "era5_swvl1", "era5_z"
]
MINMAX_COLS = [
    "era5_lsm",
    "imerg_precipitationQualityIndex",
    "imerg_probabilityLiquidPrecipitation",
]
DROP_COLS = {"era5_skt"}  # dropped here — not written to output

PASSTHROUGH_COLS = {
    "time", "latitude", "longitude",
    "is_land", "flood_composite_6h_ahead",
    "soil_no_flood_soil", "soil_average_soil",
    "soil_flood_soil", "soil_heavy_flood_soil",
    "era5_sst",  # already filled with min SST — written as-is
                 # (SST normalisation stats need to be added separately
                 #  once fill_sst_nan_v2.py has been run)
}

ALL_TRANSFORM_COLS = ZSCORE_COLS + LOG1P_ZSCORE_COLS + MINMAX_COLS
os.makedirs("eda_output", exist_ok=True)

# Pre-flight check
header_cols = pd.read_csv(SOURCE_PATH, nrows=0).columns.tolist()
missing = [c for c in ALL_TRANSFORM_COLS if c not in header_cols]
if missing:
    print(f"⚠ Columns not found in source: {missing}")
    print(f"  Run add_t2m_from_netcdf_fast.py first if era5_t2m is missing.")
    raise SystemExit(1)

present_drop = [c for c in DROP_COLS if c in header_cols]
print(f"Source:  {SOURCE_PATH} ({os.path.getsize(SOURCE_PATH)/1e9:.2f} GB)")
print(f"Output:  {OUTPUT_PATH}")
print(f"Dropping in output: {present_drop}")
print(f"Z-score:         {ZSCORE_COLS}")
print(f"Log1p + z-score: {LOG1P_ZSCORE_COLS}")
print(f"Min-max:         {MINMAX_COLS}\n")

# ── PASS 1: compute statistics ──
print("Pass 1/2: computing normalisation statistics...")
stats = {col: {"n": 0, "sum": 0.0, "sum_sq": 0.0,
               "min": np.inf, "max": -np.inf}
         for col in ALL_TRANSFORM_COLS}

n_rows = 0
for i, chunk in enumerate(pd.read_csv(SOURCE_PATH, usecols=ALL_TRANSFORM_COLS,
                                        chunksize=CHUNK_SIZE, on_bad_lines="skip")):
    n_rows += len(chunk)
    for col in ALL_TRANSFORM_COLS:
        s = chunk[col].dropna()
        if len(s) == 0:
            continue
        vals = np.log1p(s.to_numpy(dtype=np.float64).clip(0)) \
               if col in LOG1P_ZSCORE_COLS \
               else s.to_numpy(dtype=np.float64)
        stats[col]["n"]      += len(vals)
        stats[col]["sum"]    += vals.sum()
        stats[col]["sum_sq"] += (vals ** 2).sum()
        stats[col]["min"]     = min(stats[col]["min"], vals.min())
        stats[col]["max"]     = max(stats[col]["max"], vals.max())

    if (i + 1) % 20 == 0:
        print(f"  ... {n_rows:,} rows scanned")

print(f"✓ Scanned {n_rows:,} rows\n")

norm_stats = {}
for col in ALL_TRANSFORM_COLS:
    d = stats[col]
    n = d["n"]
    if n == 0:
        continue
    mean = d["sum"] / n
    var  = max(d["sum_sq"] / n - mean ** 2, 0.0)
    std  = np.sqrt(var) if var > 0 else 1.0
    norm_stats[col] = {
        "method": "log1p_zscore" if col in LOG1P_ZSCORE_COLS
                  else ("zscore" if col in ZSCORE_COLS else "minmax"),
        "mean": mean, "std": std,
        "min": d["min"], "max": d["max"],
    }
    print(f"  {col:<42} {norm_stats[col]['method']:<15} "
          f"mean={mean:.4f}  std={std:.4f}")

stats_df = pd.DataFrame([{"column": col, **v} for col, v in norm_stats.items()])
stats_df.to_csv(STATS_PATH, index=False, float_format="%.6f")
print(f"\n✓ Saved normalisation stats -> {STATS_PATH}")

# ── PASS 2: apply transforms, drop era5_skt, write output ──
print(f"\nPass 2/2: writing normalised output to {OUTPUT_PATH}...")
n_rows = 0
first_chunk = True

for i, chunk in enumerate(pd.read_csv(SOURCE_PATH, chunksize=CHUNK_SIZE,
                                        on_bad_lines="skip")):
    # Drop unwanted columns
    chunk = chunk.drop(columns=[c for c in DROP_COLS if c in chunk.columns],
                       errors="ignore")

    for col in ZSCORE_COLS:
        if col in chunk.columns:
            s = norm_stats[col]
            chunk[col] = ((chunk[col] - s["mean"]) / s["std"]).astype(np.float32)

    for col in LOG1P_ZSCORE_COLS:
        if col in chunk.columns:
            s = norm_stats[col]
            chunk[col] = (
                (np.log1p(chunk[col].clip(lower=0)) - s["mean"]) / s["std"]
            ).astype(np.float32)

    for col in MINMAX_COLS:
        if col in chunk.columns:
            s = norm_stats[col]
            denom = s["max"] - s["min"]
            if denom > 0:
                chunk[col] = (
                    (chunk[col] - s["min"]) / denom
                ).clip(0, 1).astype(np.float32)

    chunk.to_csv(OUTPUT_PATH, mode="w" if first_chunk else "a",
                 header=first_chunk, index=False)
    first_chunk = False
    n_rows += len(chunk)
    if (i + 1) % 20 == 0:
        print(f"  ... {n_rows:,} rows written")

print(f"\n✓ Wrote {n_rows:,} rows -> {OUTPUT_PATH}")
print(f"  File size: {os.path.getsize(OUTPUT_PATH)/1e9:.2f} GB")

# Spot-check
sample = pd.read_csv(OUTPUT_PATH, nrows=18340)
print("\nSpot-check (first hour):")
for col in ALL_TRANSFORM_COLS:
    if col in sample.columns:
        v = sample[col].dropna()
        print(f"  {col:<42} min={v.min():.4f}  max={v.max():.4f}  mean={v.mean():.4f}")

out_cols = pd.read_csv(OUTPUT_PATH, nrows=0).columns.tolist()
print(f"\n✓ Final columns ({len(out_cols)}): {out_cols}")
print(f"  Source unchanged: {SOURCE_PATH}")
print(f"  Normalised output: {OUTPUT_PATH}")
print(f"  Stats: {STATS_PATH}")

Source:  merged_output_hourly_imerg/merged_data_final.csv (71.11 GB)
Output:  merged_output_hourly_imerg/merged_data_normalised.csv
Dropping in output: []
Z-score:         ['era5_sst', 'era5_d2m', 'era5_t2m', 'era5_msl', 'era5_tcwv', 'era5_blh', 'wind_speed10']
Log1p + z-score: ['imerg_precipitation', 'era5_cape', 'era5_cp', 'era5_swvl1', 'era5_z']
Min-max:         ['era5_lsm', 'imerg_precipitationQualityIndex', 'imerg_probabilityLiquidPrecipitation']

Pass 1/2: computing normalisation statistics...
  ... 20,000,000 rows scanned
  ... 40,000,000 rows scanned
  ... 60,000,000 rows scanned
  ... 80,000,000 rows scanned
  ... 100,000,000 rows scanned
  ... 120,000,000 rows scanned
  ... 140,000,000 rows scanned
  ... 160,000,000 rows scanned
  ... 180,000,000 rows scanned
  ... 200,000,000 rows scanned
  ... 220,000,000 rows scanned
  ... 240,000,000 rows scanned
  ... 260,000,000 rows scanned
  ... 280,000,000 rows scanned
  ... 300,000,000 rows scanned
  ... 320,000,000 rows scanned
  .

# normalisation diagnostics

In [7]:
# SANITY CHECK — NORMALISED DATASET
# ============================================================================
# Scans merged_data_normalised.csv and prints min/max/mean/std for every
# numeric column, with pass/fail checks based on expected ranges:
#
#   Z-score columns      -> mean ≈ 0, std ≈ 1, range roughly [-5, 5]
#   Log1p + z-score      -> same expected range as z-score
#   Min-max columns      -> all values in [0, 1]
#   Binary/OHE columns   -> only 0 and 1
#   Target column        -> 0-100 (percentile rank, land cells only)
#   era5_sst             -> should have no NaN (filled with min sea SST)
# ============================================================================
import os
import numpy as np
import pandas as pd

CSV_PATH = "merged_output_hourly_imerg/merged_data_normalised.csv"
CHUNK_SIZE = 1_000_000

ZSCORE_COLS = {
    "era5_d2m", "era5_t2m", "era5_msl", "era5_tcwv",
    "era5_blh", "wind_speed10"
}
LOG1P_ZSCORE_COLS = {
    "imerg_precipitation", "era5_cape", "era5_cp", "era5_swvl1", "era5_z"
}
MINMAX_COLS = {
    "era5_lsm",
    "imerg_precipitationQualityIndex",
    "imerg_probabilityLiquidPrecipitation",
}
BINARY_COLS = {
    "is_land",
    "soil_no_flood_soil", "soil_average_soil",
    "soil_flood_soil", "soil_heavy_flood_soil",
}
TARGET_COLS = {"flood_composite_6h_ahead"}
SKIP_COLS   = {"time", "latitude", "longitude"}

print(f"Source: {CSV_PATH} ({os.path.getsize(CSV_PATH)/1e9:.2f} GB)")

header_cols = pd.read_csv(CSV_PATH, nrows=0).columns.tolist()
check_cols  = [c for c in header_cols if c not in SKIP_COLS]

print(f"Checking {len(check_cols)} columns...\n")

# Accumulate stats in one pass
stats = {col: {
    "n": 0, "sum": 0.0, "sum_sq": 0.0,
    "min": np.inf, "max": -np.inf, "n_nan": 0
} for col in check_cols}

n_rows = 0
for i, chunk in enumerate(pd.read_csv(CSV_PATH, usecols=check_cols,
                                        chunksize=CHUNK_SIZE,
                                        on_bad_lines="skip")):
    n_rows += len(chunk)
    for col in check_cols:
        s = chunk[col]
        d = stats[col]
        d["n_nan"] += int(s.isna().sum())
        valid = s.dropna()
        if len(valid) == 0:
            continue
        vals = valid.to_numpy(dtype=np.float64)
        d["n"]      += len(vals)
        d["sum"]    += vals.sum()
        d["sum_sq"] += (vals ** 2).sum()
        d["min"]     = min(d["min"], vals.min())
        d["max"]     = max(d["max"], vals.max())

    if (i + 1) % 20 == 0:
        print(f"  ... {n_rows:,} rows scanned")

print(f"✓ Scanned {n_rows:,} rows\n")

# Print results with pass/fail
print(f"{'Column':<42} {'Min':>10} {'Max':>10} {'Mean':>10} "
      f"{'Std':>8} {'NaN%':>7}  Check")
print("=" * 100)

all_passed = True
for col in check_cols:
    d = stats[col]
    n = d["n"]
    if n == 0:
        print(f"{col:<42} {'NO DATA':>10}")
        continue

    mean = d["sum"] / n
    var  = max(d["sum_sq"] / n - mean ** 2, 0.0)
    std  = np.sqrt(var)
    nan_pct = 100 * d["n_nan"] / n_rows
    lo   = d["min"]
    hi   = d["max"]

    # Determine expected range and check
    if col in ZSCORE_COLS | LOG1P_ZSCORE_COLS:
        passed = abs(mean) < 0.1 and 0.8 < std < 1.2 and lo > -10 and hi < 10
        expected = "mean≈0, std≈1, range~[-5,5]"
    elif col in MINMAX_COLS:
        passed = lo >= -0.001 and hi <= 1.001
        expected = "[0, 1]"
    elif col in BINARY_COLS:
        passed = lo >= 0 and hi <= 1 and std < 1.0
        expected = "{0, 1} binary"
    elif col in TARGET_COLS:
        passed = lo >= 0 and hi <= 100
        expected = "[0, 100] percentile"
    elif col == "era5_sst":
        passed = d["n_nan"] == 0
        expected = "no NaN (filled)"
    else:
        passed = True  # no specific expectation — just report
        expected = "report only"

    status = "✓ PASS" if passed else "✗ FAIL"
    if not passed:
        all_passed = False

    print(f"{col:<42} {lo:>10.4f} {hi:>10.4f} {mean:>10.4f} "
          f"{std:>8.4f} {nan_pct:>6.2f}%  {status}  ({expected})")

print("\n" + "=" * 100)
if all_passed:
    print("✓✓✓ ALL CHECKS PASSED — normalisation looks correct")
else:
    print("⚠ SOME CHECKS FAILED — review columns marked ✗ FAIL above")
    print("  Common causes:")
    print("  - Z-score mean far from 0: stats computed on different data subset")
    print("  - Min-max outside [0,1]: unseen values at inference exceed training range")
    print("  - NaN in era5_sst: fill_sst_nan_v2.py has not been run yet")

Source: merged_output_hourly_imerg/merged_data_normalised.csv (79.65 GB)
Checking 21 columns...

  ... 20,000,000 rows scanned
  ... 40,000,000 rows scanned
  ... 60,000,000 rows scanned
  ... 80,000,000 rows scanned
  ... 100,000,000 rows scanned
  ... 120,000,000 rows scanned
  ... 140,000,000 rows scanned
  ... 160,000,000 rows scanned
  ... 180,000,000 rows scanned
  ... 200,000,000 rows scanned
  ... 220,000,000 rows scanned
  ... 240,000,000 rows scanned
  ... 260,000,000 rows scanned
  ... 280,000,000 rows scanned
  ... 300,000,000 rows scanned
  ... 320,000,000 rows scanned
  ... 340,000,000 rows scanned
  ... 360,000,000 rows scanned
  ... 380,000,000 rows scanned
  ... 400,000,000 rows scanned
✓ Scanned 401,425,920 rows

Column                                            Min        Max       Mean      Std    NaN%  Check
era5_d2m                                      -4.9507     4.2742     0.0000   1.0000   0.00%  ✓ PASS  (mean≈0, std≈1, range~[-5,5])
era5_msl                   

In [10]:
# CHECK FLOOD CLASS BALANCE IN flood_composite_6h_ahead
# ============================================================================
# Examines the distribution of the continuous flood composite target
# on land cells only, to understand how rare flood events are before
# designing the synthetic data augmentation strategy.
# ============================================================================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CSV_PATH = "merged_output_hourly_imerg/merged_data_final.csv"
OUTPUT_DIR = "eda_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

CHUNK_SIZE = 1_000_000
TARGET_COL = "flood_composite_6h_ahead"

# Proposed 3-class thresholds (adjust if different)
THRESHOLD_AVERAGE = 80   # composite >= 80 = average flood risk
THRESHOLD_HEAVY   = 95   # composite >= 95 = heavy flood risk

print(f"Source: {CSV_PATH}")
print(f"Thresholds: average_flood >= {THRESHOLD_AVERAGE}, "
      f"heavy_flood >= {THRESHOLD_HEAVY}\n")

# Accumulate distribution on land cells only
n_no_flood   = 0
n_avg_flood  = 0
n_heavy_flood = 0
n_sea        = 0
n_nan        = 0
reservoir    = []  # reservoir sample for histogram
seen         = 0
SAMPLE_SIZE  = 500_000

import random
rng = random.Random(42)

for chunk in pd.read_csv(CSV_PATH,
                          usecols=[TARGET_COL, "is_land"],
                          chunksize=CHUNK_SIZE,
                          on_bad_lines="skip"):
    sea_mask  = chunk["is_land"] == 0
    nan_mask  = chunk[TARGET_COL].isna()
    land_valid = ~sea_mask & ~nan_mask

    n_sea += int(sea_mask.sum())
    n_nan += int((~sea_mask & nan_mask).sum())

    scores = chunk.loc[land_valid, TARGET_COL].to_numpy()
    n_no_flood    += int((scores < THRESHOLD_AVERAGE).sum())
    n_avg_flood   += int(((scores >= THRESHOLD_AVERAGE) &
                           (scores < THRESHOLD_HEAVY)).sum())
    n_heavy_flood += int((scores >= THRESHOLD_HEAVY).sum())

    # Reservoir sampling for histogram
    for v in scores:
        seen += 1
        if len(reservoir) < SAMPLE_SIZE:
            reservoir.append(float(v))
        else:
            j = rng.randint(0, seen - 1)
            if j < SAMPLE_SIZE:
                reservoir[j] = float(v)

total_land = n_no_flood + n_avg_flood + n_heavy_flood
total_all  = total_land + n_sea + n_nan

print("=" * 60)
print("FLOOD CLASS DISTRIBUTION (land cells with valid target)")
print("=" * 60)
print(f"Total rows:              {total_all:>15,}")
print(f"Sea cells (skipped):     {n_sea:>15,} ({100*n_sea/total_all:.1f}%)")
print(f"NaN target (skipped):    {n_nan:>15,} ({100*n_nan/total_all:.1f}%)")
print(f"Land cells with target:  {total_land:>15,} ({100*total_land/total_all:.1f}%)")
print()
print(f"No flood   (score < {THRESHOLD_AVERAGE}):   "
      f"{n_no_flood:>12,} ({100*n_no_flood/total_land:.2f}%)")
print(f"Avg flood  ({THRESHOLD_AVERAGE} <= score < {THRESHOLD_HEAVY}): "
      f"{n_avg_flood:>12,} ({100*n_avg_flood/total_land:.2f}%)")
print(f"Heavy flood (score >= {THRESHOLD_HEAVY}):   "
      f"{n_heavy_flood:>12,} ({100*n_heavy_flood/total_land:.2f}%)")
print()
print(f"Class imbalance ratio (no_flood : avg : heavy):")
if n_avg_flood > 0 and n_heavy_flood > 0:
    print(f"  {n_no_flood/n_heavy_flood:.0f} : "
          f"{n_avg_flood/n_heavy_flood:.0f} : 1")

# Plot distribution
arr = np.array(reservoir)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Full distribution
axes[0].hist(arr, bins=100, color="steelblue", edgecolor="none")
axes[0].axvline(THRESHOLD_AVERAGE, color="orange", linestyle="--",
                linewidth=2, label=f"avg flood threshold ({THRESHOLD_AVERAGE})")
axes[0].axvline(THRESHOLD_HEAVY, color="red", linestyle="--",
                linewidth=2, label=f"heavy flood threshold ({THRESHOLD_HEAVY})")
axes[0].set_xlabel("flood_composite_6h_ahead")
axes[0].set_ylabel("Count")
axes[0].set_title("Full distribution (land cells)")
axes[0].legend()

# Right tail zoom (flood events)
tail = arr[arr >= THRESHOLD_AVERAGE]
if len(tail) > 0:
    axes[1].hist(tail, bins=60, color="tomato", edgecolor="none")
    axes[1].axvline(THRESHOLD_HEAVY, color="darkred", linestyle="--",
                    linewidth=2, label=f"heavy flood ({THRESHOLD_HEAVY})")
    axes[1].set_xlabel("flood_composite_6h_ahead")
    axes[1].set_title(f"Right tail zoom (score >= {THRESHOLD_AVERAGE}, "
                      f"n={len(tail):,})")
    axes[1].legend()

fig.suptitle("Flood composite score distribution — class balance check",
             fontsize=13)
fig.tight_layout()
out_path = os.path.join(OUTPUT_DIR, "flood_class_balance.png")
fig.savefig(out_path, dpi=140, bbox_inches="tight")
plt.close(fig)
print(f"\n✓ Saved distribution plot -> {out_path}")

Source: merged_output_hourly_imerg/merged_data_final.csv
Thresholds: average_flood >= 80, heavy_flood >= 95

FLOOD CLASS DISTRIBUTION (land cells with valid target)
Total rows:                  401,425,920
Sea cells (skipped):         290,716,416 (72.4%)
NaN target (skipped):             30,348 (0.0%)
Land cells with target:      110,679,156 (27.6%)

No flood   (score < 80):     86,882,572 (78.50%)
Avg flood  (80 <= score < 95):   20,581,852 (18.60%)
Heavy flood (score >= 95):      3,214,732 (2.90%)

Class imbalance ratio (no_flood : avg : heavy):
  27 : 6 : 1

✓ Saved distribution plot -> eda_output/flood_class_balance.png


In [11]:
# EDA ON CHUNKS — SUMMARY STATISTICS (NO CONCATENATION)
# ============================================================================
# Computes per-column statistics from merged_data_final.csv in a single
# chunked pass, with no full-file load into memory. Uses reservoir sampling
# for unbiased percentile estimates across the full 400M-row dataset.
# ============================================================================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import random

CSV_PATH = "merged_output_hourly_imerg/merged_data_final.csv"
OUTPUT_DIR = "eda_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

EXCLUDE_COLS = {"time", "date", "latitude", "longitude",
                "era5_lat_idx", "era5_lon_idx"}
HIST_BINS = 60
SAMPLE_SIZE = 200_000
CHUNK_SIZE = 500_000

print(f"Source: {CSV_PATH} ({os.path.getsize(CSV_PATH)/1e9:.2f} GB)")
print("Computing statistics from CSV (chunked, no concatenation)...\n")

stats_dict = {}
total_rows = 0
rng = random.Random(42)

for i, chunk in enumerate(pd.read_csv(CSV_PATH, chunksize=CHUNK_SIZE,
                                        on_bad_lines="skip")):
    total_rows += len(chunk)
    if (i + 1) % 20 == 0:
        print(f"  ... {total_rows:,} rows processed")

    for col in chunk.columns:
        if col in EXCLUDE_COLS or not pd.api.types.is_numeric_dtype(chunk[col]):
            continue
        if col not in stats_dict:
            stats_dict[col] = {
                "count": 0, "sum": 0.0, "sum_sq": 0.0,
                "min": np.inf, "max": -np.inf, "missing": 0,
                "reservoir": [], "seen": 0,
            }
        s = chunk[col]
        valid = s.dropna()
        d = stats_dict[col]
        d["missing"] += int(s.isna().sum())

        if len(valid) == 0:
            continue

        d["count"] += len(valid)
        d["sum"] += float(valid.sum())
        d["sum_sq"] += float((valid ** 2).sum())
        d["min"] = min(d["min"], float(valid.min()))
        d["max"] = max(d["max"], float(valid.max()))

        vals = valid.to_numpy()
        for v in vals:
            d["seen"] += 1
            if len(d["reservoir"]) < SAMPLE_SIZE:
                d["reservoir"].append(float(v))
            else:
                j = rng.randint(0, d["seen"] - 1)
                if j < SAMPLE_SIZE:
                    d["reservoir"][j] = float(v)

print(f"\n✓ Scanned {total_rows:,} rows. Building summary table...")

rows = []
for col, d in stats_dict.items():
    count = d["count"]
    mean = d["sum"] / count if count > 0 else np.nan
    var = (d["sum_sq"] / count - mean ** 2) if count > 0 else np.nan
    std = float(np.sqrt(max(var, 0))) if count > 0 else np.nan

    res = np.sort(np.array(d["reservoir"])) if d["reservoir"] else np.array([])
    p25 = float(np.percentile(res, 25)) if len(res) > 0 else np.nan
    p50 = float(np.percentile(res, 50)) if len(res) > 0 else np.nan
    p75 = float(np.percentile(res, 75)) if len(res) > 0 else np.nan
    p95 = float(np.percentile(res, 95)) if len(res) > 0 else np.nan

    rows.append({
        "attribute":  col,
        "count":      count,
        "missing":    int(d["missing"]),
        "missing_%":  round(100 * d["missing"] / max(count + d["missing"], 1), 3),
        "min":        round(d["min"], 4) if count > 0 else np.nan,
        "p25":        round(p25, 4),
        "median":     round(p50, 4),
        "mean":       round(mean, 4) if count > 0 else np.nan,
        "p75":        round(p75, 4),
        "p95":        round(p95, 4),
        "max":        round(d["max"], 4) if count > 0 else np.nan,
        "std":        round(std, 4) if count > 0 else np.nan,
    })

stats_df = pd.DataFrame(rows).sort_values("attribute").reset_index(drop=True)

stats_csv = os.path.join(OUTPUT_DIR, "summary_statistics.csv")
stats_df.to_csv(stats_csv, index=False, float_format="%.4f")
print(f"✓ Saved summary statistics -> {stats_csv}")
print(f"\n{stats_df.to_string(index=False)}")

print("\nGenerating summary table image...")
fig, ax = plt.subplots(figsize=(min(2 + 1.1 * stats_df.shape[1], 22),
                                 0.5 + 0.4 * stats_df.shape[0]))
ax.axis("off")
tbl = ax.table(
    cellText=stats_df.values,
    colLabels=stats_df.columns,
    cellLoc="center",
    loc="center",
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(7)
tbl.scale(1, 1.3)
for (r, c), cell in tbl.get_celld().items():
    if r == 0:
        cell.set_facecolor("#34568B")
        cell.set_text_props(color="white", fontweight="bold")
    elif r % 2 == 0:
        cell.set_facecolor("#f0f3f7")

ax.set_title("Summary statistics per attribute", fontsize=12, pad=12)
fig.tight_layout()
table_path = os.path.join(OUTPUT_DIR, "summary_statistics.png")
fig.savefig(table_path, dpi=140, bbox_inches="tight")
plt.close(fig)
print(f"✓ Saved summary table image -> {table_path}")
print(f"\n✓ EDA complete. Outputs in: {OUTPUT_DIR}/")

Source: merged_output_hourly_imerg/merged_data_final.csv (71.11 GB)
Computing statistics from CSV (chunked, no concatenation)...

  ... 10,000,000 rows processed
  ... 20,000,000 rows processed
  ... 30,000,000 rows processed
  ... 40,000,000 rows processed
  ... 50,000,000 rows processed
  ... 60,000,000 rows processed
  ... 70,000,000 rows processed
  ... 80,000,000 rows processed
  ... 90,000,000 rows processed
  ... 100,000,000 rows processed
  ... 110,000,000 rows processed
  ... 120,000,000 rows processed
  ... 130,000,000 rows processed
  ... 140,000,000 rows processed
  ... 150,000,000 rows processed
  ... 160,000,000 rows processed
  ... 170,000,000 rows processed
  ... 180,000,000 rows processed
  ... 190,000,000 rows processed
  ... 200,000,000 rows processed
  ... 210,000,000 rows processed
  ... 220,000,000 rows processed
  ... 230,000,000 rows processed
  ... 240,000,000 rows processed
  ... 250,000,000 rows processed
  ... 260,000,000 rows processed
  ... 270,000,000 row